# Module 3: Skills + Steering (15 min)

Add workflow knowledge via **skills** (markdown documents the agent can activate) and enforce business rules with **steering handlers** — a deterministic refund workflow enforcer and an LLM-based tone guardrail.

**Prerequisites:** Modules 1-2 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Skills — Workflow Knowledge

Skills are markdown files (`SKILL.md`) that give the agent step-by-step procedures. The agent loads them on demand via the `AgentSkills` plugin.

Check out the `skills/` folder — it has three skills:
- `refund-processing` — how to handle refund requests
- `order-tracking` — how to check order status
- `account-troubleshooting` — how to handle account issues

In [ ]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests. When a customer needs help, activate the appropriate
skill for step-by-step guidance.

Important guidelines:
- Always ask for the customer ID first if you don't have it.
- Use the data returned by tools to answer questions.
- Be warm but efficient."""

skills_plugin = AgentSkills(skills=["./skills"])

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[skills_plugin],
    system_prompt=SYSTEM_PROMPT,
)

# The agent will activate the refund-processing skill to guide its workflow
result = agent("Hi, I'm customer C-1001. I'd like a refund for my wireless headphones.")

---

## Part 2: Steering Handlers — Deterministic Enforcement

Skills are suggestions. Steering handlers are **enforcement**. The `RefundWorkflowHandler` blocks `process_refund` unless the agent has already called `lookup_customer` and `get_order_history`.

In [ ]:
import re
from strands.vended_plugins.steering import (
    SteeringHandler, LLMSteeringHandler,
    Proceed, Guide, ToolSteeringAction,
    LedgerProvider,
)


class RefundWorkflowHandler(SteeringHandler):
    """Deterministic handler: enforce refund workflow ordering.

    Rules:
    1. Must look up the customer before processing a refund
    2. Must check order history before processing a refund
    """

    name = "refund-workflow"

    def __init__(self):
        super().__init__(context_providers=[LedgerProvider()])

    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "process_refund":
            return Proceed(reason="Not a refund operation")

        print("[STEERING] 🔍 process_refund attempted — checking workflow...")

        # Get execution history through managed ledger
        ledger = self.steering_context.data.get("ledger", {})
        tool_calls = ledger.get("tool_calls", [])

        # Must look up customer first
        customer_verified = any(
            c["tool_name"] == "lookup_customer" and c["status"] == "success"
            for c in tool_calls
        )
        if not customer_verified:
            print("[STEERING] ⚠️  Blocked: must look up customer first")
            return Guide(
                reason="You must look up the customer with lookup_customer "
                "before processing a refund."
            )

        # Must check order history
        order_checked = any(
            c["tool_name"] == "get_order_history" and c["status"] == "success"
            for c in tool_calls
        )
        if not order_checked:
            print("[STEERING] ⚠️  Blocked: must check order history first")
            return Guide(
                reason="You must check the order history with get_order_history "
                "before processing a refund."
            )

        print("[STEERING] ✅ Refund workflow validated — proceeding")
        return Proceed(reason="Refund workflow validated")


print("✅ RefundWorkflowHandler defined")

---

## Part 3: LLM-Based Steering — Tone Guardrail

The `ToneGuardrailHandler` uses a second LLM call to evaluate the agent's response quality after each model turn.

In [ ]:
class ToneGuardrailHandler(LLMSteeringHandler):
    """LLM-based handler: evaluates response tone and professionalism."""

    name = "tone-guardrail"

    def __init__(self):
        super().__init__(
            system_prompt="""You are evaluating a customer service agent's responses.
Ensure the agent follows these communication guidelines:

- Don't overpromise or guarantee specific timelines beyond what the system confirms
- Don't blame the customer, other departments, or third parties
- Acknowledge the customer's frustration before jumping to solutions
- Keep responses concise and actionable — no walls of text
- Never share internal system details or technical jargon with the customer

If the agent violates any of these, provide specific guidance on what to fix."""
        )

    async def steer_after_model(self, **kwargs):
        print("[TONE] 🔍 Evaluating agent response...")
        result = await super().steer_after_model(**kwargs)
        action_type = type(result).__name__
        print(f"[TONE] {'✅ Approved' if action_type == 'Proceed' else '⚠️  Guided'}")
        return result


tone_handler = ToneGuardrailHandler()
print("✅ ToneGuardrailHandler defined")

---

## Part 4: Full Agent with Skills + Steering

Combine everything: tools, skills, and both steering handlers.

In [ ]:
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[
        AgentSkills(skills=["./skills"]),
        RefundWorkflowHandler(),
        tone_handler,
    ],
    system_prompt=SYSTEM_PROMPT,
)

# The steering handler will enforce: lookup → orders → refund
result = agent(
    "I'm customer C-1001. I want a refund for my wireless headphones order ORD-5521. "
    "Yes, please process it."
)

---

## 🎯 Try It Yourself

Test the steering enforcement — what happens if you try to skip steps?

In [ ]:
# Try asking for a refund without providing a customer ID
# The steering handler should force the agent to look up the customer first

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[RefundWorkflowHandler(), tone_handler],
    system_prompt="You are a customer service agent. Process refunds immediately when asked.",
)

# Even with a permissive prompt, steering blocks the shortcut
result = agent("Process a refund for order ORD-5521, amount $79.99")

---

## What's Next

The agent is smart and safe, but it forgets everything between sessions. In **Module 4: Session Managers**, you'll add persistence so the agent remembers previous conversations.